# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [56]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

from sklearn.ensemble import VotingClassifier
import itertools

from sklearn.ensemble import BaggingClassifier

from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [4]:
df_not_scaled = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_target = pd.read_csv('../data/dayofweek.csv')
df_not_scaled['dayofweek'] = df_target['dayofweek']

X = df_not_scaled.drop('dayofweek', axis=1)
y = df_not_scaled['dayofweek']

### first split

In [5]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

### second split

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full
)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [9]:
def print_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    
    print(f"accuracy is {acc:.5f}")
    print(f"precision is {prec:.5f}")
    print(f"recall is {rec:.5f}")

In [91]:
svc = SVC(
    C=10,
    class_weight=None,
    gamma='auto', 
    kernel='rbf', 
    probability=True, 
    random_state=21
)
svc.fit(X_train, y_train)
print_metrics(y_valid, svc.predict(X_valid))

accuracy is 0.87778
precision is 0.88162
recall is 0.87778


In [26]:
dt = DecisionTreeClassifier(
    max_depth=21, 
    criterion='gini',
    class_weight='balanced',
    random_state=21
)
dt.fit(X_train, y_train)
print_metrics(y_valid, dt.predict(X_valid))

accuracy is 0.86667
precision is 0.87170
recall is 0.86667


In [25]:
rf = RandomForestClassifier(
    class_weight=None,
    n_estimators=100, 
    max_depth=24,
    criterion='entropy',
    random_state=21,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print_metrics(y_valid, rf.predict(X_valid))

accuracy is 0.90000
precision is 0.89969
recall is 0.90000


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [80]:
estimators = [
    ('svm', svc),
    ('dt', dt),
    ('rf', rf)
]

# комбинации возможных весов
weights_options = [
    (1, 1, 1),
    (1, 2, 1),
    (2, 2, 1),
    (1, 2, 2),
    (4, 1, 4)
]


In [81]:
best_acc = 0
best_weights = None
best_model = None

for w in weights_options:
    voter = VotingClassifier(estimators=estimators, voting='soft', weights=w)
    voter.fit(X_train, y_train)

    acc = accuracy_score(y_valid, voter.predict(X_valid))
    print(f"Weights {w}: Accuracy = {acc:.5f}")

    if acc > best_acc:
        best_acc = acc
        best_weights = w
        best_model = voter

print(f"\nBest Weights: {best_weights} with Valid Accuracy: {best_acc:.5f}")

Weights (1, 1, 1): Accuracy = 0.87778
Weights (1, 2, 1): Accuracy = 0.86667
Weights (2, 2, 1): Accuracy = 0.87037
Weights (1, 2, 2): Accuracy = 0.87407
Weights (4, 1, 4): Accuracy = 0.90000

Best Weights: (4, 1, 4) with Valid Accuracy: 0.90000


In [85]:
y_test_pred = best_model.predict(X_test)
print_metrics(y_test, y_test_pred)

accuracy is 0.90828
precision is 0.91214
recall is 0.90828


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [101]:
n_estimators_list = [3, 5, 50]
best_acc = 0
best_n = 0
best_bagging = None

In [102]:
for n in n_estimators_list:
    bagging = BaggingClassifier(
        estimator=svc,
        n_estimators=n,
        random_state=21,
        n_jobs=-1
    )
    bagging.fit(X_train, y_train)

    acc = accuracy_score(y_valid, bagging.predict(X_valid))
    print(f"n_estimators={n}: Accuracy = {acc:.5f}")

    if acc > best_acc:
        best_acc = acc
        best_n = n
        best_bagging = bagging

print(f"\nBest n_estimators: {best_n} (Valid Accuracy: {best_acc:.5f})")

n_estimators=3: Accuracy = 0.86667
n_estimators=5: Accuracy = 0.87407
n_estimators=50: Accuracy = 0.88148

Best n_estimators: 50 (Valid Accuracy: 0.88148)


In [37]:
y_test_pred = best_bagging.predict(X_test)
print_metrics(y_test, y_test_pred)

accuracy is 0.87278
precision is 0.87840
recall is 0.87278


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [40]:
estimators = [
    ('svm', svc),
    ('dt', dt),
    ('rf', rf)
]

In [123]:
n_splits_list = [2, 4, 7]
passthrough_list = [False, True]

In [124]:
best_acc = 0
best_params = {}
best_stacking = None

In [125]:
for n in n_splits_list:
    for pass_bool in passthrough_list:
        cv_strat = StratifiedKFold(n_splits=n, shuffle=True, random_state=21)

        final_layer = OneVsRestClassifier(LogisticRegression(solver='liblinear', random_state=21))

        stack = StackingClassifier(
            estimators=estimators,
            final_estimator=final_layer,
            cv=cv_strat,
            passthrough=pass_bool,
            n_jobs=-1
        )   
        
        stack.fit(X_train, y_train)

        acc = accuracy_score(y_valid, stack.predict(X_valid))
        print(f"n_splits={n}, Passthrough={pass_bool}: Accuracy = {acc:.5f}")
        
        if acc > best_acc:
            best_acc = acc
            best_params = {'n_splits': n, 'passthrough': pass_bool}
            best_stacking = stack

n_splits=2, Passthrough=False: Accuracy = 0.90000
n_splits=2, Passthrough=True: Accuracy = 0.90000
n_splits=4, Passthrough=False: Accuracy = 0.90741
n_splits=4, Passthrough=True: Accuracy = 0.90741
n_splits=7, Passthrough=False: Accuracy = 0.90370
n_splits=7, Passthrough=True: Accuracy = 0.90370


In [126]:
print(f"Best Params: {best_params} (Valid Accuracy: {best_acc:.5f})")

Best Params: {'n_splits': 4, 'passthrough': False} (Valid Accuracy: 0.90741)


In [128]:
y_test_pred = best_stacking.predict(X_test)
print()
print_metrics(y_test, y_test_pred)



accuracy is 0.90828
precision is 0.91159
recall is 0.90828


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [51]:
final_ensemble = best_stacking

In [68]:
df_analysis = pd.read_csv('../data/day-of-week-not-scaled.csv')
target = pd.read_csv('../data/dayofweek.csv')['dayofweek']

X_all = df_analysis.copy()

all_preds = final_ensemble.predict(X_all)

df_analysis['dayofweek'] = target
df_analysis['predicted'] = all_preds
df_analysis['is_error'] = df_analysis['dayofweek'] != df_analysis['predicted']


df_analysis['uid'] = df_analysis.filter(like='uid_').idxmax(axis=1).str.replace('uid_', '')

df_analysis['labname'] = df_analysis.filter(like='labname_').idxmax(axis=1).str.replace('labname_', '')

def get_error_stats(df, col):
    return df.groupby(col)['is_error'].agg(['sum', 'count']).assign(
        error_rate=lambda x: x['sum'] / x['count'] * 100
    ).sort_values('error_rate', ascending=False)

In [70]:
print(get_error_stats(df_analysis, 'dayofweek'))

           sum  count  error_rate
dayofweek                        
0           13    136    9.558824
5           13    271    4.797048
1           12    274    4.379562
2            6    149    4.026846
6            9    356    2.528090
4            2    104    1.923077
3            4    396    1.010101


In [71]:
print(get_error_stats(df_analysis, 'labname').head(5))

         sum  count  error_rate
labname                        
lab03      1      1  100.000000
laba06s    6     61    9.836066
laba04s   10    104    9.615385
lab05s     3     36    8.333333
laba06     3     48    6.250000


In [72]:
print(get_error_stats(df_analysis, 'uid').head(5))

         sum  count  error_rate
uid                            
user_22    3      7   42.857143
user_23    1      4   25.000000
user_6     3     12   25.000000
user_17    3     34    8.823529
user_2     9    121    7.438017


In [73]:
joblib.dump(final_ensemble, '../data/final_ensemble.joblib')

['../data/final_ensemble.joblib']